# ESA CCI CMIP7 recipe example

This notebook uses a tiny synthetic CMIP7-style ESA CCI water-vapour dataset and a discovered recipe to show workflow-style reformatting with Woodpecker.


In [ ]:
import numpy as np
import woodpecker_cmip7_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_cmip7

Create a deterministic ESA CCI-like input: `prw` should become `tcwv`, `bnds` should become `nv`, selected metadata should be added, and latitude should be increasing.

In [ ]:
source_name = "ESACCI-WATERVAPOUR-L3C-TCWV-meris-005deg-2002-2017-fv3.2.zarr"
dataset = make_cmip7(variable="prw", overrides={"source_name": source_name}, seed=7)
dataset = dataset.isel(lat=slice(None, None, -1))
dataset = dataset.assign_coords(bnds=[0, 1])
dataset["lat_bnds"] = (
    ("lat", "bnds"),
    np.column_stack([dataset["lat"].values - 0.5, dataset["lat"].values + 0.5]),
)

dataset

Load the discovered ESA CCI recipe and inspect its content.

The recipe is bundled in the CMIP7 plugin package.


In [ ]:
recipe = woodpecker.recipe.get("cmip7.esa_cci_water_vapour_zarr")

recipe.model_dump()

In [ ]:
recipe.match.model_dump(), [step.id for step in recipe.steps]

In [ ]:
findings = woodpecker.recipe.check(dataset, recipe)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
preview = woodpecker.recipe.fix(dataset, recipe, dry_run=True)

preview.stats, preview.preview, tuple(dataset.data_vars), tuple(dataset.dims)

Apply the same recipe in memory with `dry_run=False`.

In [ ]:
write = woodpecker.recipe.fix(dataset, recipe, dry_run=False)

(
    write.stats,
    tuple(dataset.data_vars),
    tuple(dataset.dims),
    dataset.attrs["realm"],
    dataset.attrs["branded_variable"],
    float(dataset["lat"].values[0]) < float(dataset["lat"].values[-1]),
)

In [ ]:
recheck = woodpecker.recipe.check(dataset, recipe)
bool(recheck)